# 04 — Player Archetype Clustering
**NBA Player Performance & Trade Value Predictor**  
DATA 300 · Spring 2026 · Anh Vu & Drew Norton

---

### Goals of this notebook
1. Load and prepare the cleaned player-season dataset
2. Select and scale features for clustering
3. Find the optimal number of clusters `k` using the **elbow method** and **silhouette scores**
4. Fit a final **K-Means** model and label archetypes
5. Validate with **Hierarchical Clustering** (Adjusted Rand Index)
6. Visualize clusters in 2D using **PCA**
7. Export cluster assignments for use in the trade value notebook


---
## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

# Consistent plot style
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
})

RANDOM_STATE = 42
print('Imports OK')

---
## 1. Load Data

Expects `data/processed/players_combined.csv` produced by `02_feature_engineering.ipynb`.  
Minimum required columns: `player`, `season`, `PER`, `TS_pct`, `BPM`, `VORP`, `ast_pct`, `reb_pct`, `blk_pct`, `stl_pct`, `tov_pct`.

In [ ]:
df = pd.read_csv('data/processed/players_combined.csv')
print(f'Loaded {len(df):,} player-season records')
print(f'Seasons: {df["season"].min()} – {df["season"].max()}')
df.head(3)

---
## 2. Feature Selection & Scaling

We cluster on **role-defining** stats rather than volume stats (points, rebounds, assists).  
Volume stats are biased by minutes played and would just separate starters from bench players.  
Instead we use **rate/efficiency** metrics that capture *how* a player contributes.

| Feature | Why it's included |
|---|---|
| `PER` | Overall efficiency |
| `TS_pct` | Shooting efficiency (accounts for 3s and FTs) |
| `BPM` | Box plus/minus — impact on scoring margin |
| `VORP` | Value over replacement — best single value metric |
| `ast_pct` | Playmaking / creation rate |
| `reb_pct` | Rebounding involvement |
| `blk_pct` | Rim protection |
| `stl_pct` | Perimeter defense / activity |
| `tov_pct` | Ball security (higher = worse) |

In [ ]:
CLUSTER_FEATURES = [
    'PER', 'TS_pct', 'BPM', 'VORP',
    'ast_pct', 'reb_pct', 'blk_pct', 'stl_pct', 'tov_pct'
]

# Drop rows with any missing values in cluster features
df_clean = df.dropna(subset=CLUSTER_FEATURES).copy()
print(f'Rows after dropping NaNs: {len(df_clean):,} (dropped {len(df) - len(df_clean):,})')

X = df_clean[CLUSTER_FEATURES].values

# StandardScaler: zero mean, unit variance — critical because PER (~15) and TS% (~0.55) are on very different scales
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('\nFeature means before scaling:', X.mean(axis=0).round(2))
print('Feature means after scaling: ', X_scaled.mean(axis=0).round(2))  # should all be ~0

---
## 3. Finding the Optimal k

We use **two complementary methods** — never rely on just one:

- **Elbow Method (WCSS/Inertia):** look for the "elbow" where adding more clusters stops meaningfully reducing inertia
- **Silhouette Score:** measures how well each point fits its own cluster vs. the nearest neighboring cluster. Range: [-1, 1], higher = better

In [ ]:
K_RANGE = range(2, 13)
inertias = []
silhouette_scores = []

print('Testing k values...')
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=15, max_iter=500)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil)
    print(f'  k={k:2d} | inertia={km.inertia_:,.1f} | silhouette={sil:.4f}')

print('Done.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# --- Elbow plot ---
ax1.plot(list(K_RANGE), inertias, 'o-', color='#1F3864', linewidth=2, markersize=6)
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia (WCSS)', fontsize=12)
ax1.set_title('Elbow Method', fontsize=13, fontweight='bold')
ax1.set_xticks(list(K_RANGE))
ax1.grid(axis='y', alpha=0.3)

# --- Silhouette plot ---
best_k_sil = list(K_RANGE)[silhouette_scores.index(max(silhouette_scores))]
colors_sil = ['#E8A020' if k == best_k_sil else '#1F3864' for k in K_RANGE]
ax2.bar(list(K_RANGE), silhouette_scores, color=colors_sil, edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Number of Clusters (k)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score by k', fontsize=13, fontweight='bold')
ax2.set_xticks(list(K_RANGE))
ax2.grid(axis='y', alpha=0.3)
ax2.annotate(f'Best: k={best_k_sil}', xy=(best_k_sil, max(silhouette_scores)),
             xytext=(best_k_sil + 0.8, max(silhouette_scores) - 0.01),
             fontsize=10, color='#E8A020', fontweight='bold')

plt.suptitle('Choosing Optimal k for Player Archetype Clustering', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/clustering_k_selection.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'\nSilhouette peaks at k={best_k_sil}')

---
## 4. Fit Final K-Means Model

Based on the plots above, choose your `K_FINAL`.  
Our slides propose **k=6** archetypes — confirm this matches the elbow/silhouette results and adjust if needed.

In [ ]:
# ---- ADJUST THIS if your elbow/silhouette suggest a different k ----
K_FINAL = 6
# -------------------------------------------------------------------

km_final = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=20, max_iter=500)
df_clean['cluster_kmeans'] = km_final.fit_predict(X_scaled)

final_sil = silhouette_score(X_scaled, df_clean['cluster_kmeans'])
print(f'Final model | k={K_FINAL} | silhouette={final_sil:.4f}')
print(f'Cluster sizes:\n{df_clean["cluster_kmeans"].value_counts().sort_index()}')

### 4a. Inspect Cluster Centroids
This is how you **name** the archetypes — look at which features are high/low per cluster.

In [ ]:
centroid_df = df_clean.groupby('cluster_kmeans')[CLUSTER_FEATURES].mean().round(3)
centroid_df['n_players'] = df_clean['cluster_kmeans'].value_counts().sort_index()

# Highlight max value in each column to make patterns obvious
display(centroid_df.style.highlight_max(axis=0, color='#fff3cd').format(precision=3))

### 4b. Assign Archetype Labels

After inspecting the centroids above, map each cluster number to an archetype name.  
**Update the dictionary below to match what you observe.**

In [ ]:
# ---- UPDATE THESE based on your centroid inspection ----
# Pattern hints:
#   Elite Scorer    → high PER, high BPM, high VORP, high TS%
#   3-and-D Wing    → high TS%, high stl_pct, moderate ast_pct, low tov_pct
#   Defensive Anchor → high blk_pct, high reb_pct, low ast_pct
#   Playmaking Guard → high ast_pct, high stl_pct, low reb_pct
#   Stretch Big     → high TS%, high reb_pct, moderate blk_pct
#   Utility Bench   → below-average across all metrics

ARCHETYPE_MAP = {
    0: 'Elite Scorer',
    1: '3-and-D Wing',
    2: 'Defensive Anchor',
    3: 'Playmaking Guard',
    4: 'Stretch Big',
    5: 'Utility Bench',
}
# --------------------------------------------------------

df_clean['archetype'] = df_clean['cluster_kmeans'].map(ARCHETYPE_MAP)

# Spot-check: show top players per archetype by PER
for arch in ARCHETYPE_MAP.values():
    top = df_clean[df_clean['archetype'] == arch].nlargest(3, 'PER')[['player', 'season', 'PER']]
    print(f"\n{arch}:")
    print(top.to_string(index=False))

---
## 5. Silhouette Analysis — Per-Sample

This plot shows how well **each individual player** fits their cluster.  
Wide, uniform bars = tight, well-separated clusters.  
Negative values = players that probably belong in a neighboring cluster.

In [ ]:
ARCHETYPE_COLORS = {
    'Elite Scorer':     '#E8A020',
    '3-and-D Wing':     '#2E8B57',
    'Defensive Anchor': '#4682B4',
    'Playmaking Guard': '#9370DB',
    'Stretch Big':      '#DC143C',
    'Utility Bench':    '#888888',
}

sample_sil = silhouette_samples(X_scaled, df_clean['cluster_kmeans'])
df_clean['silhouette_val'] = sample_sil

fig, ax = plt.subplots(figsize=(10, 7))
y_lower = 10

for cluster_id in sorted(df_clean['cluster_kmeans'].unique()):
    arch_name = ARCHETYPE_MAP[cluster_id]
    cluster_sil = sample_sil[df_clean['cluster_kmeans'] == cluster_id]
    cluster_sil_sorted = np.sort(cluster_sil)
    size = len(cluster_sil_sorted)
    y_upper = y_lower + size
    color = ARCHETYPE_COLORS[arch_name]
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil_sorted,
                     facecolor=color, edgecolor=color, alpha=0.8)
    ax.text(-0.05, y_lower + size / 2, arch_name, ha='right', va='center', fontsize=9)
    y_lower = y_upper + 10

ax.axvline(x=final_sil, color='black', linestyle='--', linewidth=1.5,
           label=f'Avg silhouette = {final_sil:.3f}')
ax.set_xlabel('Silhouette Coefficient', fontsize=12)
ax.set_title('Per-Sample Silhouette Analysis by Archetype', fontsize=13, fontweight='bold')
ax.set_yticks([])
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('figures/silhouette_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 6. Validate with Hierarchical Clustering

Hierarchical clustering makes no assumptions about cluster shape or initialization.  
If it agrees with K-Means, our archetypes are **robust**.  
We measure agreement using the **Adjusted Rand Index (ARI)**:  
- ARI = 1.0 → perfect agreement  
- ARI = 0.0 → random (no agreement)  
- ARI > 0.6 → strong structural agreement

In [ ]:
hier = AgglomerativeClustering(n_clusters=K_FINAL, linkage='ward')
df_clean['cluster_hier'] = hier.fit_predict(X_scaled)

ari = adjusted_rand_score(df_clean['cluster_kmeans'], df_clean['cluster_hier'])
print(f'Adjusted Rand Index (K-Means vs Hierarchical): {ari:.4f}')

if ari > 0.6:
    print('✅ Strong agreement — cluster structure is robust')
elif ari > 0.4:
    print('⚠️  Moderate agreement — clusters are reasonably stable')
else:
    print('❌ Low agreement — consider revisiting k or feature selection')

In [ ]:
# Cross-tabulation: shows how hierarchical clusters map onto K-Means clusters
# Ideally each hierarchical cluster aligns strongly with one K-Means cluster
crosstab = pd.crosstab(
    df_clean['cluster_hier'],
    df_clean['archetype'],
    rownames=['Hierarchical'],
    colnames=['K-Means Archetype']
)
display(crosstab.style.background_gradient(cmap='Blues', axis=None))

---
## 7. PCA Visualization — Cluster Biplot

PCA reduces our 9 features to 2 dimensions for visualization.  
Good clusters will show **clear separation** in this 2D space.  
This plot goes directly on the **poster**.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
df_clean['PC1'] = coords[:, 0]
df_clean['PC2'] = coords[:, 1]

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
print(f'PC1 explains {var1:.1f}% of variance')
print(f'PC2 explains {var2:.1f}% of variance')
print(f'Total explained: {var1 + var2:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

for arch_name, color in ARCHETYPE_COLORS.items():
    mask = df_clean['archetype'] == arch_name
    ax.scatter(
        df_clean[mask]['PC1'],
        df_clean[mask]['PC2'],
        label=arch_name,
        color=color,
        alpha=0.6,
        s=25,
        edgecolors='white',
        linewidths=0.3
    )

# Annotate a handful of well-known players for readability
NOTABLE_PLAYERS = ['LeBron James', 'Stephen Curry', 'Rudy Gobert',
                   'Chris Paul', 'Karl-Anthony Towns', 'Mikal Bridges']
for _, row in df_clean[df_clean['player'].isin(NOTABLE_PLAYERS)].drop_duplicates('player').iterrows():
    ax.annotate(row['player'], (row['PC1'], row['PC2']),
                fontsize=7.5, ha='left', va='bottom',
                xytext=(4, 4), textcoords='offset points',
                color='#222222')

ax.set_xlabel(f'PC1 ({var1:.1f}% variance explained)', fontsize=12)
ax.set_ylabel(f'PC2 ({var2:.1f}% variance explained)', fontsize=12)
ax.set_title('NBA Player Archetypes — PCA Cluster Biplot (2015–2024)',
             fontsize=14, fontweight='bold')
ax.legend(title='Archetype', title_fontsize=10, fontsize=9,
          loc='upper right', framealpha=0.85)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.4)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.4)

plt.tight_layout()
plt.savefig('figures/pca_cluster_biplot.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → figures/pca_cluster_biplot.png')

### 7b. PCA Feature Loading Plot
Shows which original features drive PC1 and PC2 — helpful for interpreting what the axes mean.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=CLUSTER_FEATURES,
    columns=['PC1', 'PC2']
).sort_values('PC1')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, pc in zip([ax1, ax2], ['PC1', 'PC2']):
    sorted_load = loadings[pc].sort_values()
    colors = ['#DC143C' if v < 0 else '#1F3864' for v in sorted_load]
    ax.barh(sorted_load.index, sorted_load.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{pc} Feature Loadings', fontsize=12, fontweight='bold')
    ax.set_xlabel('Loading coefficient')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('What drives each PCA component?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/pca_loadings.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 8. Archetype Profile Radar Chart
A radar/spider chart showing the average stat profile of each archetype.  
Great for the **poster** — makes archetype differences immediately intuitive.

In [ ]:
# Use a subset of features that are most interpretable for a radar
RADAR_FEATURES = ['PER', 'TS_pct', 'ast_pct', 'reb_pct', 'blk_pct', 'stl_pct', 'BPM']
N = len(RADAR_FEATURES)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

# Normalize each feature to [0,1] across the cluster means
centroid_radar = df_clean.groupby('archetype')[RADAR_FEATURES].mean()
centroid_norm = (centroid_radar - centroid_radar.min()) / (centroid_radar.max() - centroid_radar.min())

fig, axes = plt.subplots(2, 3, figsize=(14, 9), subplot_kw=dict(polar=True))
axes = axes.flatten()

for i, (arch_name, color) in enumerate(ARCHETYPE_COLORS.items()):
    ax = axes[i]
    values = centroid_norm.loc[arch_name].tolist()
    values += values[:1]  # close
    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(RADAR_FEATURES, fontsize=9)
    ax.set_yticklabels([])
    ax.set_title(arch_name, fontsize=11, fontweight='bold', pad=15, color=color)
    ax.set_ylim(0, 1)
    ax.grid(color='gray', alpha=0.3)

plt.suptitle('NBA Player Archetype Profiles', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/archetype_radar.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → figures/archetype_radar.png')

---
## 9. Export Results
Save cluster assignments to disk so `05_trade_value.ipynb` can join them in.

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)
os.makedirs('figures', exist_ok=True)

output_cols = ['player', 'season', 'cluster_kmeans', 'archetype',
               'silhouette_val', 'PC1', 'PC2'] + CLUSTER_FEATURES

df_clean[output_cols].to_csv('data/processed/players_clustered.csv', index=False)

print(f'Saved {len(df_clean):,} rows to data/processed/players_clustered.csv')
print('\nArchetype distribution:')
print(df_clean['archetype'].value_counts().to_string())
print(f'\nOverall silhouette score: {final_sil:.4f}')
print(f'ARI (K-Means vs Hierarchical): {ari:.4f}')

---
## Summary

| Step | What we did | Key output |
|---|---|---|
| Feature selection | 9 efficiency/rate stats | `X_scaled` |
| k selection | Elbow + silhouette | Optimal `K_FINAL` |
| K-Means | Fit final model | `cluster_kmeans`, `archetype` |
| Silhouette analysis | Per-sample fit quality | `silhouette_analysis.png` |
| Hierarchical validation | ARI agreement check | ARI score |
| PCA biplot | 2D visualization | `pca_cluster_biplot.png` |
| Radar chart | Archetype profiles | `archetype_radar.png` |
| Export | Cluster assignments for trade value | `players_clustered.csv` |

**Next:** `05_trade_value.ipynb` — combine cluster archetypes with regression predictions and salary data to compute the final trade value score.